In [1]:
%%spark
--conf spark.jars.packages=com.microsoft.ml.spark:mmlspark_2.11:0.18.1
--conf spark.jars.ivySettings=/home/sankuai/.m2/ivysettings.xml
--conf spark.yarn.queue=root.zw03.hadoop-jiulvalgo.etl
--conf spark.executor.cores=4
--conf spark.task.cpus=4

,Driver日志,队列信息
,/home/sankuai/logs/spark-1783929219.log,root.zw03.hadoop-jiulvalgo.etl


CmdOutput(['tail', '-50', '/home/sankuai/logs/spark-1783929219.log'], 2)

:: loading settings :: file = /home/sankuai/.m2/ivysettings.xml


Ivy Default Cache set to: /home/sankuai/.ivy2/cache
The jars for the packages stored in: /home/sankuai/.ivy2/jars
:: loading settings :: url = jar:file:/opt/meituan/spark-3.0/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml
com.microsoft.ml.spark#mmlspark_2.11 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d8fa766a-bfc8-4fff-ba5c-1ac0edb33a07;1.0
	confs: [default]
	found com.microsoft.ml.spark#mmlspark_2.11;0.18.1 in mtdp
	found org.scalactic#scalactic_2.11;3.0.5 in mtdp
	found org.scala-lang#scala-reflect;2.11.12 in mtdp
	found org.scalatest#scalatest_2.11;3.0.5 in mtdp
	found org.scala-lang.modules#scala-xml_2.11;1.0.6 in mtdp
	found io.spray#spray-json_2.11;1.3.2 in mtdp
	found com.microsoft.cntk#cntk;2.4 in mtdp
	found org.openpnp#opencv;3.2.0-1 in mtdp
	found com.jcraft#jsch;0.1.54 in mtdp
	found org.apache.httpcomponents#httpclient;4.5.6 in mtdp
	found org.apache.httpcomponents#httpcore;4.4.10 in mtdp
	found commons-logging

,SparkSession,SparkContext,Job Search/applicationId
链接/变量名,spark,sc,application_1780904968055_5289215


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib
import os, subprocess
import optuna

import lightgbm as lgb
import xgboost as xgb
from lightgbm import early_stopping, log_evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, roc_auc_score, classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, log_loss
from sklearn.isotonic import IsotonicRegression

from pyspark.sql import functions as F
from pyspark.sql.functions import expr, when, col, udf, avg, lit, row_number
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

In [3]:
%%sql df --preview --quiet

SELECT a.*,b.intention
FROM mart_jiulv_flow.dws_stp_session_comp_df a
left join mart_hoteldim.stp_test_session_v2 b
on a.session_id = b.session_id
where a.dt between 20260301 and 20260630

/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

,session_id,holiday_weekend_vacation_checkin_datekey_cnt,holiday_checkin_datekey_cnt,weekend_checkin_datekey_cnt,vacation_checkin_datekey_cnt,lead_time,checkin_datekey_cnt,goods_cnt,poi_cnt,aoi_cnt,city_cnt,stay_time,search_page_cnt,search_page_time,poi_list_cnt,poi_list_time,poi_info_cnt,poi_info_time,room_list_cnt,room_list_time,room_info_cnt,room_info_time,create_order_cnt,create_order_time,evaluate_page_cnt,evaluate_page_time,search_scenic_cnt,waitou_cnt,rednote_cnt,order_cnt,star2_cnt,star3_cnt,star4_cnt,star5_cnt,budget_cnt,express_cnt,business_cnt,theme_cnt,couple_cnt,apartment_cnt,inn_cnt,homestay_cnt,hostel_cnt,farmstay_cnt,family_guesthouse_cnt,guesthouse_cnt,resort_hotel_cnt,villa_cnt,family_cnt,esports_cnt,scarce_cnt,unique_cnt,lowprice_cnt,good_cnt,tourism_core_city_cnt,tourism_big_city_cnt,tourism_seasonal_city_cnt,tier1_city_cnt,new_tier1_city_cnt,tier2_city_cnt,tier3_city_cnt,tier4_city_cnt,tier5_city_cnt,university_cnt,transportation_hub_cnt,scenic_area_cnt,hospital_cnt,performance_sports_venue_cnt,convention_center_cnt,industrial_park_cnt,king_room_cnt,single_room_cnt,double_room_cnt,triple_room_cnt,suite_cnt,standalone_cnt,dorm_cnt,is_college_student,is_adult_single,is_adult_married_no_kids,is_adult_married_with_kids,is_over_60,is_senior,is_middle_age,is_young,is_minor,is_high_value,is_mid_value,is_low_value,is_l4,is_l1_3,is_youth_campus,is_business_elite,is_self_social,is_practical_life,is_family_guardian,is_vital_soldier,is_quality_visitor,is_kid_explorer,is_occasional_traveler,holiday_rnt_pct,travel_distance,is_holiday_weekend_vacation,is_holiday,is_weekend,is_vacation,order_lead_time,is_star2,is_star3,is_star4,is_star5,is_budget,is_express,is_business,is_theme,is_couple,is_apartment,is_inn,is_homestay,is_hostel,is_farmstay,is_family_guesthouse,is_guesthouse,is_resort_hotel,is_villa,is_family,is_esports,is_scarce,is_unique,is_lowprice,is_good,order_travel_distance,is_search_scenic,is_waitou,is_rednote,is_tourism_core_city,is_tourism_big_city,is_tourism_seasonal_city,is_tier1_city,is_new_tier1_city,is_tier2_city,is_tier3_city,is_tier4_city,is_tier5_city,is_university,is_transportation_hub,is_scenic_area,is_hospital,is_performance_sports_venue,is_convention_center,is_industrial_park,is_king_room,is_single_room,is_double_room,is_triple_room,is_suite,is_standalone,is_dorm,search_cnt,travel_session_cnt,is_search,is_travel_session,session_cnt,event_cnt,advance_checkin_datekey_cnt,sameday_checkin_datekey_cnt,overnight_checkin_datekey_cnt,dt,intention
0,0004d9c2-d945-4555-bb2b-3ed4cb40dffe1774328469906206,0,0,0,0,1.968586,3,10,33,5,2,2016.590,10,192.233,81,1012.949,55,518.295,0,0.0,0,0.0,5,0.000,0,0.0,0,0,0,7,89,4,0,0,66,2,2,1,0,5,0,3,0,0,0,0,0,0,2,0,0,2,88,79,0,93,0,92,1,0,0,0,0,70,0,22,91,69,0,0,34,0,3,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.428571,671.340350,0,0,0,0,2.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,1,1,668,0,1,0,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,67,0,1,0,5,235,1,2,0,20260324,NaN
1,0013B6F7-9F93-474A-AE7F-9EE4732608161774322092873217,1,0,1,0,0.768868,4,20,19,4,2,2139.327,3,674.675,65,655.400,83,508.624,0,0.0,0,0.0,8,32.564,0,0.0,0,181,0,5,90,20,3,4,32,49,9,0,0,11,0,12,0,0,0,0,4,0,11,8,4,17,70,82,0,27,0,0,27,0,90,0,0,90,3,3,24,0,0,0,10,4,8,4,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.024060,28.780815,0,0,0,0,0.0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,28,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,109,0,1,0,11,226,2,4,0,20260324,NaN
2,001B52C0-649D-4B69-95A4-EF1B6CCF44B71774308721889368,0,0,0,0,4.908046,2,4,16,10,2,795.935,10,238.896,34,264.949,23,146.796,0,0.0,0,0.0,3,12.991,0,0.0,0,100,45,0,18,3,3,3,5,0,10,0,0,9,0,0,0,0,0,0,0,0,4,1,1,3,23,27,0,18,9,18,0,0,0,9,0,13,12,7,0,6,2,3,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.125000,150.878609,0,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0,0,5,102,

In [4]:
print("df records read: " + str(df.count()))
print("df columns read: " , len(df.columns))
print("df columns: ", df.columns)

df records read: 1771621511
df columns read:  169
df columns:  ['session_id', 'holiday_weekend_vacation_checkin_datekey_cnt', 'holiday_checkin_datekey_cnt', 'weekend_checkin_datekey_cnt', 'vacation_checkin_datekey_cnt', 'lead_time', 'checkin_datekey_cnt', 'goods_cnt', 'poi_cnt', 'aoi_cnt', 'city_cnt', 'stay_time', 'search_page_cnt', 'search_page_time', 'poi_list_cnt', 'poi_list_time', 'poi_info_cnt', 'poi_info_time', 'room_list_cnt', 'room_list_time', 'room_info_cnt', 'room_info_time', 'create_order_cnt', 'create_order_time', 'evaluate_page_cnt', 'evaluate_page_time', 'search_scenic_cnt', 'waitou_cnt', 'rednote_cnt', 'order_cnt', 'star2_cnt', 'star3_cnt', 'star4_cnt', 'star5_cnt', 'budget_cnt', 'express_cnt', 'business_cnt', 'theme_cnt', 'couple_cnt', 'apartment_cnt', 'inn_cnt', 'homestay_cnt', 'hostel_cnt', 'farmstay_cnt', 'family_guesthouse_cnt', 'guesthouse_cnt', 'resort_hotel_cnt', 'villa_cnt', 'family_cnt', 'esports_cnt', 'scarce_cnt', 'unique_cnt', 'lowprice_cnt', 'good_cnt', 'to

In [3]:
def metrics(df):
    # time
    df = df.withColumn('workday_cnt',col('checkin_datekey_cnt')-col('holiday_weekend_vacation_checkin_datekey_cnt'))
    df = df.withColumn('is_workday',1-col('is_holiday_weekend_vacation'))
    df = df.withColumn("is_advance_booking",when(col("order_lead_time") > 0, 1).otherwise(0))
    df = df.withColumn("is_sameday_booking",when(col("order_lead_time") == 0, 1).otherwise(0))
    df = df.withColumn("is_overnight_booking",when(col("order_lead_time") < 0, 1).otherwise(0))
    df = df.withColumn("is_holiday_weekend_vacation_advance_booking",when((col('holiday_weekend_vacation_checkin_datekey_cnt')>0)&(col('advance_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_holiday_weekend_vacation_sameday_booking",when((col('holiday_weekend_vacation_checkin_datekey_cnt')>0)&(col('sameday_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_holiday_weekend_vacation_overnight_booking",when((col('holiday_weekend_vacation_checkin_datekey_cnt')>0)&(col('overnight_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_workday_advance_booking",when((col('workday_cnt')>0)&(col('advance_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_workday_sameday_booking",when((col('workday_cnt')>0)&(col('sameday_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_workday_overnight_booking",when((col('workday_cnt')>0)&(col('overnight_checkin_datekey_cnt')>0),1).otherwise(0))
    df = df.withColumn("is_order_holiday_weekend_vacation_advance_booking",when((col('is_holiday_weekend_vacation')==1)&(col('is_advance_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_holiday_weekend_vacation_sameday_booking",when((col('is_holiday_weekend_vacation')==1)&(col('is_sameday_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_holiday_weekend_vacation_overnight_booking",when((col('is_holiday_weekend_vacation')==1)&(col('is_overnight_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_workday_advance_booking",when((col('is_workday')==1)&(col('is_advance_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_workday_sameday_booking",when((col('is_workday')==1)&(col('is_sameday_booking')==1),1).otherwise(0))
    df = df.withColumn("is_order_workday_overnight_booking",when((col('is_workday')==1)&(col('is_overnight_booking')==1),1).otherwise(0))
    
    # distance
    df = df.withColumn("is_long_distance",when(col('travel_distance')>1000,1).otherwise(0))
    df = df.withColumn("is_mid_distance",when((col('travel_distance')>301)&(col('travel_distance')<=1000),1).otherwise(0))
    df = df.withColumn("is_nearby_distance",when((col('travel_distance')>51)&(col('travel_distance')<=300),1).otherwise(0))
    df = df.withColumn("is_short_distance",when(col('travel_distance')<=50,1).otherwise(0))
    df = df.withColumn("is_order_long_distance",when(col('order_travel_distance')>1000,1).otherwise(0))
    df = df.withColumn("is_order_mid_distance",when((col('order_travel_distance')>301)&(col('order_travel_distance')<=1000),1).otherwise(0))
    df = df.withColumn("is_order_nearby_distance",when((col('order_travel_distance')>51)&(col('order_travel_distance')<=300),1).otherwise(0))
    df = df.withColumn("is_order_short_distance",when(col('order_travel_distance')<=50,1).otherwise(0))
    
    # city
    df = df.withColumn('high_tier_city_cnt',col('tier1_city_cnt') +col('new_tier1_city_cnt') +col('tier2_city_cnt'))
    df = df.withColumn('low_tier_city_cnt',col('tier3_city_cnt') +col('tier4_city_cnt') +col('tier5_city_cnt'))
    df = df.withColumn('high_tier_city_ratio',when(col('city_cnt') > 0,col('high_tier_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('low_tier_city_ratio',when(col('city_cnt') > 0,col('low_tier_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('is_high_tier_city',when((col('is_tier1_city')==1)|(col('is_new_tier1_city')==1)|(col('is_tier2_city')==1),1).otherwise(0))
    df = df.withColumn('is_low_tier_city',when((col('is_tier3_city')==1)|(col('is_tier4_city')==1)|(col('is_tier5_city')==1),1).otherwise(0))
    df = df.withColumn('tourism_core_city_ratio',when(col('city_cnt') > 0,col('tourism_core_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('tourism_big_city_ratio',when(col('city_cnt') > 0,col('tourism_big_city_cnt') / col('city_cnt')).otherwise(0))
    df = df.withColumn('tourism_seasonal_city_ratio',when(col('city_cnt') > 0,col('tourism_seasonal_city_cnt') / col('city_cnt')).otherwise(0))
    
    # aoi
    df = df.withColumn('university_ratio', when(col('aoi_cnt')>0, col('university_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('transportation_hub_ratio', when(col('aoi_cnt')>0, col('transportation_hub_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('scenic_area_ratio', when(col('aoi_cnt')>0, col('scenic_area_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('hospital_ratio', when(col('aoi_cnt')>0, col('hospital_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('performance_sports_venue_ratio', when(col('aoi_cnt')>0, col('performance_sports_venue_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('convention_center_ratio', when(col('aoi_cnt')>0, col('convention_center_cnt') / col('aoi_cnt')).otherwise(0))
    df = df.withColumn('industrial_park_ratio', when(col('aoi_cnt')>0, col('industrial_park_cnt') / col('aoi_cnt')).otherwise(0))
    
    # poi
    df = df.withColumn('high_star_cnt',col('star3_cnt') +col('star4_cnt') +col('star5_cnt'))
    df = df.withColumn('is_high_star',when((col('is_star3')==1)|(col('is_star4')==1)|(col('is_star5')==1),1).otherwise(0))
    df = df.withColumn('high_star_ratio',when(col('poi_cnt') > 0,col('high_star_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('low_star_ratio',when(col('poi_cnt') > 0,col('star3_cnt') / col('poi_cnt')).otherwise(0))
    
    df = df.withColumn('budget_ratio', when(col('poi_cnt')>0, col('budget_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('express_ratio', when(col('poi_cnt')>0, col('express_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('business_ratio', when(col('poi_cnt')>0, col('business_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('theme_ratio', when(col('poi_cnt')>0, col('theme_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('couple_ratio', when(col('poi_cnt')>0, col('couple_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('apartment_ratio', when(col('poi_cnt')>0, col('apartment_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('inn_ratio', when(col('poi_cnt')>0, col('inn_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('homestay_ratio', when(col('poi_cnt')>0, col('homestay_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('hostel_ratio', when(col('poi_cnt')>0, col('hostel_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('farmstay_ratio', when(col('poi_cnt')>0, col('farmstay_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('family_guesthouse_ratio', when(col('poi_cnt')>0, col('family_guesthouse_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('guesthouse_ratio', when(col('poi_cnt')>0, col('guesthouse_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('resort_hotel_ratio', when(col('poi_cnt')>0, col('resort_hotel_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('villa_ratio', when(col('poi_cnt')>0, col('villa_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('family_ratio', when(col('poi_cnt')>0, col('family_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('esports_ratio', when(col('poi_cnt')>0, col('esports_cnt') / col('poi_cnt')).otherwise(0))
    
    df = df.withColumn('scarce_ratio', when(col('poi_cnt')>0, col('scarce_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('unique_ratio', when(col('poi_cnt')>0, col('unique_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('lowprice_ratio', when(col('poi_cnt')>0, col('lowprice_cnt') / col('poi_cnt')).otherwise(0))
    df = df.withColumn('good_ratio', when(col('poi_cnt')>0, col('good_cnt') / col('poi_cnt')).otherwise(0))
    
    # goods
    df = df.withColumn('king_room_ratio', when(col('goods_cnt')>0, col('king_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('single_room_ratio', when(col('goods_cnt')>0, col('single_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('double_room_ratio', when(col('goods_cnt')>0, col('double_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('triple_room_ratio', when(col('goods_cnt')>0, col('triple_room_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('suite_ratio', when(col('goods_cnt')>0, col('suite_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('standalone_ratio', when(col('goods_cnt')>0, col('standalone_cnt') / col('goods_cnt')).otherwise(0))
    df = df.withColumn('dorm_ratio', when(col('goods_cnt')>0, col('dorm_cnt') / col('goods_cnt')).otherwise(0))
    
    # browsing
    df = df.withColumn('avg_checkin_datekey_cnt',when(col('session_cnt')>0,col('checkin_datekey_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_city_cnt',when(col('session_cnt')>0,col('city_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_aoi_cnt',when(col('session_cnt')>0,col('aoi_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_poi_cnt',when(col('session_cnt')>0,col('poi_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_goods_cnt',when(col('session_cnt')>0,col('goods_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_stay_time',when(col('session_cnt')>0,col('stay_time')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_order_cnt',when(col('session_cnt')>0,col('order_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_search_cnt',when(col('session_cnt')>0,col('search_cnt')/col('session_cnt')).otherwise(0))
    df = df.withColumn('avg_travel_session_cnt',when(col('session_cnt')>0,col('travel_session_cnt')/col('session_cnt')).otherwise(0))
    
    df = df.withColumn('search_page_time_ratio',when(col('stay_time')>0,col('search_page_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('poi_list_time_ratio',when(col('stay_time')>0,col('poi_list_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('poi_info_time_ratio',when(col('stay_time')>0,col('poi_info_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('room_list_time_ratio',when(col('stay_time')>0,col('room_list_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('room_info_time_ratio',when(col('stay_time')>0,col('room_info_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('create_order_time_ratio',when(col('stay_time')>0,col('create_order_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('evaluate_page_time_ratio',when(col('stay_time')>0,col('evaluate_page_time')/col('stay_time')).otherwise(0))
    df = df.withColumn('search_page_cnt_ratio',when(col('event_cnt')>0,col('search_page_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('poi_list_cnt_ratio',when(col('event_cnt')>0,col('poi_list_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('poi_info_cnt_ratio',when(col('event_cnt')>0,col('poi_info_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('room_list_cnt_ratio',when(col('event_cnt')>0,col('room_list_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('room_info_cnt_ratio',when(col('event_cnt')>0,col('room_info_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('create_order_cnt_ratio',when(col('event_cnt')>0,col('create_order_cnt')/col('event_cnt')).otherwise(0))
    df = df.withColumn('evaluate_page_cnt_ratio',when(col('event_cnt')>0,col('evaluate_page_cnt')/col('event_cnt')).otherwise(0))
    
    df = df.withColumn('search_non_scenic_cnt',col('search_cnt')-col('search_scenic_cnt'))
    df = df.withColumn('search_scenic_ratio',when(col('search_cnt')>0,col('search_scenic_cnt')/col('search_cnt')).otherwise(0))
    df = df.withColumn('search_non_scenic_ratio',when(col('search_cnt')>0,col('search_non_scenic_cnt')/col('search_cnt')).otherwise(0))
    
    return df

In [17]:
data = metrics(df)
data = data.dropna(subset=['intention'])
data = data.toPandas()

Exception in thread "serve-DataFrame" java.net.SocketTimeoutException: Accept timed out
	at java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:404)
	at java.net.ServerSocket.implAccept(ServerSocket.java:545)
	at java.net.ServerSocket.accept(ServerSocket.java:513)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:58)
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

data records read:  1017337
data columns read:  243
data columns:  Index(['session_id', 'holiday_weekend_vacation_checkin_datekey_cnt',
       'holiday_checkin_datekey_cnt', 'weekend_checkin_datekey_cnt',
       'vacation_checkin_datekey_cnt', 'lead_time', 'checkin_datekey_cnt',
       'goods_cnt', 'poi_cnt', 'aoi_cnt',
       ...
       'search_page_ratio', 'poi_list_ratio', 'poi_info_ratio',
       'room_list_ratio', 'room_info_ratio', 'create_order_ratio',
       'evaluate_page_ratio', 'search_non_scenic_cnt', 'search_scenic_ratio',
       'search_non_scenic_ratio'],
      dtype='object', length=243)


In [16]:
# data.to_parquet("hdfs:///user/luyiwei03/stp_data.parquet", index=False)
data = pd.read_parquet("hdfs:///user/luyiwei03/stp_data_v2.parquet")

In [5]:
print("data records read: ", len(data))
print("data columns read: " , len(data.columns))
print("data columns: ", data.columns)

data records read:  181469
data columns read:  268
data columns:  Index(['session_id', 'holiday_weekend_vacation_checkin_datekey_cnt',
       'holiday_checkin_datekey_cnt', 'weekend_checkin_datekey_cnt',
       'vacation_checkin_datekey_cnt', 'lead_time', 'checkin_datekey_cnt',
       'goods_cnt', 'poi_cnt', 'aoi_cnt',
       ...
       'search_page_cnt_ratio', 'poi_list_cnt_ratio', 'poi_info_cnt_ratio',
       'room_list_cnt_ratio', 'room_info_cnt_ratio', 'create_order_cnt_ratio',
       'evaluate_page_cnt_ratio', 'search_non_scenic_cnt',
       'search_scenic_ratio', 'search_non_scenic_ratio'],
      dtype='object', length=268)


In [6]:
corr = data.corr(numeric_only=True)['intention'].sort_values(ascending=False)
print(corr)

intention                                              1.000000
is_scenic_area                                         0.327522
is_holiday_weekend_vacation_advance_booking            0.265214
is_family_guesthouse                                   0.259861
holiday_weekend_vacation_checkin_datekey_cnt           0.255713
is_travel_session                                      0.252020
is_holiday_weekend_vacation                            0.243991
weekend_checkin_datekey_cnt                            0.239166
aoi_cnt                                                0.235329
is_advance_booking                                     0.231392
is_order_holiday_weekend_vacation_advance_booking      0.229348
advance_checkin_datekey_cnt                            0.227381
poi_cnt                                                0.224518
scenic_area_cnt                                        0.223790
poi_info_cnt_ratio                                     0.220877
search_scenic_ratio                     

In [17]:
X = data.drop(columns=['intention','session_id','dt','level3_biz_code'])
y = data['intention']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=3, stratify=y)

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [10]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'num_leaves': 256,
    'max_depth': 12,
    'min_data_in_leaf': 150,
    'lambda_l1': 0.1,
    'lambda_l2': 2,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'min_gain_to_split': 0,
    'scale_pos_weight': 1.2,
    'max_bin': 255,
    'verbose': -1,
    'seed': 0
}

lgbm = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    valid_names=['train','valid'],
    num_boost_round=10000,
    callbacks=[early_stopping(stopping_rounds=200), log_evaluation(100)]
)

Training until validation scores don't improve for 200 rounds
[100]	train's auc: 0.903601	valid's auc: 0.890518
[200]	train's auc: 0.915306	valid's auc: 0.898573
[300]	train's auc: 0.924407	valid's auc: 0.904585
[400]	train's auc: 0.931578	valid's auc: 0.908861
[500]	train's auc: 0.936925	valid's auc: 0.911816
[600]	train's auc: 0.941532	valid's auc: 0.91383
[700]	train's auc: 0.945284	valid's auc: 0.915358
[800]	train's auc: 0.948743	valid's auc: 0.916373
[900]	train's auc: 0.951956	valid's auc: 0.917165
[1000]	train's auc: 0.95493	valid's auc: 0.917906
[1100]	train's auc: 0.95722	valid's auc: 0.918364
[1200]	train's auc: 0.959404	valid's auc: 0.918735
[1300]	train's auc: 0.961927	valid's auc: 0.919107
[1400]	train's auc: 0.963923	valid's auc: 0.919309
[1500]	train's auc: 0.965851	valid's auc: 0.919571
[1600]	train's auc: 0.967714	valid's auc: 0.919794
[1700]	train's auc: 0.969431	valid's auc: 0.919949
[1800]	train's auc: 0.971093	valid's auc: 0.920135
[1900]	train's auc: 0.972536	val

In [11]:
y_pred_prob = lgbm.predict(X_test)

cal_prob_raw = lgbm.predict(X_test,num_iteration=lgbm.best_iteration)

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(cal_prob_raw,y_test)

test_prob_raw = lgbm.predict(X_test,num_iteration=lgbm.best_iteration)
test_prob_cal = iso.transform(test_prob_raw)

print("Raw AUC:", roc_auc_score(y_test, test_prob_raw))
print("Calibrated AUC:", roc_auc_score(y_test, test_prob_cal))
print("Raw LogLoss:", log_loss(y_test, test_prob_raw))
print("Calibrated LogLoss:", log_loss(y_test, test_prob_cal))

best_acc=0
best_threshold=0.5
for t in np.arange(0.1, 0.9, 0.01):
    pred = (test_prob_cal >= t).astype(int)
    acc = accuracy_score(y_test, pred)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

y_pred = (test_prob_cal >= best_threshold).astype(int)

print("Best threshold:",best_threshold)
print("Best Accuracy:",best_acc)
print("AUC:",roc_auc_score(y_test,y_pred_prob))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Raw AUC: 0.9210823368607524
Calibrated AUC: 0.9218444406386397
Raw LogLoss: 0.3573860529686011
Calibrated LogLoss: 0.3515797365291188
Best threshold: 0.4999999999999998
Best Accuracy: 0.8416267151595305
AUC: 0.9210823368607524
              precision    recall  f1-score   support

           0       0.81      0.84      0.82      7983
           1       0.87      0.84      0.86     10164

    accuracy                           0.84     18147
   macro avg       0.84      0.84      0.84     18147
weighted avg       0.84      0.84      0.84     18147

[[6714 1269]
 [1605 8559]]


In [12]:
gain = lgbm.feature_importance(importance_type='gain')
features = lgbm.feature_name()

fi = pd.DataFrame({
    'feature': features,
    'gain': gain
}).sort_values(by='gain', ascending=False)

print(fi)

                                                 feature          gain
64                                       scenic_area_cnt  1.089679e+06
91                                     is_business_elite  5.288892e+05
218                                       homestay_ratio  2.740864e+05
202                                    scenic_area_ratio  2.726106e+05
40                                          homestay_cnt  2.325320e+05
100                                      travel_distance  2.068126e+05
99                                       holiday_rnt_pct  1.783443e+05
246                               avg_travel_session_cnt  1.593129e+05
101                          is_holiday_weekend_vacation  1.388366e+05
128                                          is_lowprice  1.154931e+05
82                                         is_middle_age  1.073046e+05
133                                           is_rednote  1.069106e+05
256                                   poi_info_cnt_ratio  1.051937e+05
175   

In [13]:
lgbm.save_model('lgb_intention_model_v2.txt')
joblib.dump(iso,'lgb_intention_model_v2_iso.pkl')
with open('lgb_intention_model_v2.txt.meta', 'w') as f:
    json.dump({'best_threshold': best_threshold}, f)

In [27]:
dtrain = xgb.DMatrix(X_train,label=y_train)
dval = xgb.DMatrix(X_test,label=y_test)

In [29]:
params = {
    'objective':'binary:logistic',
    'eval_metric':'auc',
    'booster':'gbtree',
    'learning_rate':0.02,
    'max_depth':10,
    'min_child_weight':50,
    'reg_alpha':1,
    'reg_lambda':5,
    'colsample_bytree':0.7,
    'subsample':0.9,
    'gamma':0.1,
    'scale_pos_weight':0.8,
    'tree_method':'hist',
    'max_bin':255,
    'seed':0,
    'n_jobs':-1
}

xgbm = xgb.train(
    params,
    dtrain,
    num_boost_round=10000,
    evals=[(dtrain,'train'),(dval,'valid')],
    early_stopping_rounds=200,
    verbose_eval=100
)

[0]	train-auc:0.85330	valid-auc:0.84387
[100]	train-auc:0.90753	valid-auc:0.89419
[200]	train-auc:0.92023	valid-auc:0.90467
[300]	train-auc:0.92742	valid-auc:0.91001
[400]	train-auc:0.93208	valid-auc:0.91285
[500]	train-auc:0.93585	valid-auc:0.91475
[600]	train-auc:0.93892	valid-auc:0.91589
[700]	train-auc:0.94149	valid-auc:0.91667
[800]	train-auc:0.94381	valid-auc:0.91720
[900]	train-auc:0.94594	valid-auc:0.91764
[1000]	train-auc:0.94796	valid-auc:0.91795
[1100]	train-auc:0.94975	valid-auc:0.91827
[1200]	train-auc:0.95146	valid-auc:0.91847
[1300]	train-auc:0.95311	valid-auc:0.91872
[1400]	train-auc:0.95462	valid-auc:0.91885
[1500]	train-auc:0.95607	valid-auc:0.91898
[1600]	train-auc:0.95749	valid-auc:0.91908
[1700]	train-auc:0.95889	valid-auc:0.91916
[1800]	train-auc:0.96025	valid-auc:0.91922
[1900]	train-auc:0.96140	valid-auc:0.91926
[2000]	train-auc:0.96257	valid-auc:0.91931
[2100]	train-auc:0.96372	valid-auc:0.91941
[2200]	train-auc:0.96470	valid-auc:0.91945
[2300]	train-auc:0.9657

In [30]:
cal_prob_raw = xgbm.predict(dval,iteration_range=(0,xgbm.best_iteration+1))

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(cal_prob_raw,y_test)

test_prob_raw = xgbm.predict(dval,iteration_range=(0,xgbm.best_iteration+1))
test_prob_cal = iso.transform(test_prob_raw)

print("Raw AUC:", roc_auc_score(y_test, test_prob_raw))
print("Calibrated AUC:", roc_auc_score(y_test, test_prob_cal))
print("Raw LogLoss:", log_loss(y_test, test_prob_raw))
print("Calibrated LogLoss:", log_loss(y_test, test_prob_cal))

best_acc=0
best_threshold=0.5
for t in np.arange(0.1, 0.9, 0.01):
    pred = (test_prob_cal >= t).astype(int)
    acc = accuracy_score(y_test, pred)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

y_pred = (test_prob_cal >= best_threshold).astype(int)

print("Best threshold:",best_threshold)
print("Best Accuracy:",best_acc)
print("AUC:",roc_auc_score(y_test,y_pred_prob))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Raw AUC: 0.9196558255951512
Calibrated AUC: 0.9204857141082415
Raw LogLoss: 0.3596105441149252
Calibrated LogLoss: 0.35435714312451794
Best threshold: 0.48999999999999977
Best Accuracy: 0.8416818206866149
AUC: 0.9201769792883865
              precision    recall  f1-score   support

           0       0.81      0.84      0.82      7983
           1       0.87      0.84      0.86     10164

    accuracy                           0.84     18147
   macro avg       0.84      0.84      0.84     18147
weighted avg       0.84      0.84      0.84     18147

[[6686 1297]
 [1576 8588]]


In [31]:
gain = xgbm.get_score(importance_type='gain')
features = X_train.columns

fi = pd.DataFrame({
    'feature': features,
    'gain': [gain.get(f,0) for f in features]
}).sort_values(by='gain', ascending=False)

print(fi)

                                                 feature        gain
91                                     is_business_elite  352.958954
40                                          homestay_cnt  161.451904
64                                       scenic_area_cnt  152.809616
167                                           is_workday  110.756592
246                               avg_travel_session_cnt  107.028740
120                                 is_family_guesthouse  103.704536
181                     is_order_workday_sameday_booking  101.168335
84                                              is_minor  100.408813
128                                          is_lowprice   96.025513
175                           is_workday_sameday_booking   93.554993
158                                   travel_session_cnt   91.806244
145                                       is_scenic_area   74.048965
25                                     search_scenic_cnt   69.024277
218                               

In [32]:
xgbm.save_model('xgb_intention_model_v2.json')
joblib.dump(iso,'xgb_intention_model_v2_iso.pkl')
with open('xgb_intention_model_v2.txt.meta', 'w') as f:
    json.dump({'best_threshold': best_threshold}, f)

In [7]:
lgbm = lgb.Booster(model_file='lgb_intention_model_v2.txt')
iso_lgbm = joblib.load('lgb_intention_model_v2_iso.pkl')
with open('lgb_intention_model_v2.txt.meta', 'r') as f:
    meta = json.load(f)
best_threshold_lgbm = meta['best_threshold']

In [8]:
xgbm = xgb.Booster(model_file='xgb_intention_model_v2.json')
iso_xgbm = joblib.load('xgb_intention_model_v2_iso.pkl')
with open('xgb_intention_model_v2.txt.meta', 'r') as f:
    meta = json.load(f)
best_threshold_xgbm = meta['best_threshold']

In [18]:
lgb_prob_raw = lgbm.predict(X_test)
lgb_prob_cal = iso_lgbm.transform(lgb_prob_raw)

dtest = xgb.DMatrix(X_test)
xgb_prob_raw = xgbm.predict(dtest)
xgb_prob_cal = iso_xgbm.transform(xgb_prob_raw)

print("LGBM Raw AUC:",roc_auc_score(y_test,lgb_prob_raw))
print("LGBM Cal AUC:",roc_auc_score(y_test,lgb_prob_cal))
print("XGB Raw AUC:",roc_auc_score(y_test,xgb_prob_raw))
print("XGB Cal AUC:",roc_auc_score(y_test,xgb_prob_cal))

best_threshold = 0.5
best_acc = 0
best_weight = 0

for w in np.arange(0,1.01,0.05):
    ensemble_prob = w*lgb_prob_cal+(1-w)*xgb_prob_cal
    for t in np.arange(0.1,0.91,0.01):
        pred = (ensemble_prob>=t).astype(int)
        acc = accuracy_score(y_test,pred)
        if acc > best_acc:
            best_acc = acc
            best_weight = w
            best_threshold = t

y_pred = (ensemble_prob >= best_threshold).astype(int)

print("Best weight:",best_weight)
print("Best threshold:",best_threshold)
print("Best Accuracy:",best_acc)
print("AUC:",roc_auc_score(y_test,ensemble_prob))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

LGBM Raw AUC: 0.9210823368607524
LGBM Cal AUC: 0.9218444406386397
XGB Raw AUC: 0.9637207470045431
XGB Cal AUC: 0.9635148046544992
Best weight: 0.0
Best threshold: 0.5299999999999998
Best Accuracy: 0.9009202623023089
AUC: 0.9218444406386397
              precision    recall  f1-score   support

           0       0.81      0.84      0.82      7983
           1       0.87      0.84      0.86     10164

    accuracy                           0.84     18147
   macro avg       0.84      0.84      0.84     18147
weighted avg       0.84      0.84      0.84     18147

[[6740 1243]
 [1631 8533]]


In [21]:
ensemble_meta = {
    "lgb_weight": float(best_weight),
    "xgb_weight": float(1-best_weight),
    "best_threshold": float(best_threshold)
}
with open('lgb_xgb_ensemble.meta', 'w') as f:
    json.dump(ensemble_meta, f)

In [23]:
with open('lgb_xgb_ensemble.meta','r') as f:
    ensemble_meta = json.load(f)
lgb_weight = ensemble_meta['lgb_weight']
xgb_weight = ensemble_meta['xgb_weight']
best_threshold_ensemble = ensemble_meta['best_threshold']

In [7]:
%%sql rela --preview --quiet

select *
from mart_jiulv_flow.dim_stp_session_rel_df

,session_id,related_session_id,dt,level3_biz_code
0,09134BB6-4A1B-4B55-BF39-6FC24963369A1759568750251996,09134BB6-4A1B-4B55-BF39-6FC24963369A1759240160230285,20251004,hotel
1,340fd86c-afd9-4f08-ac51-33c5e788f6df1759569819029113,340fd86c-afd9-4f08-ac51-33c5e788f6df1759569867084794,20251004,hotel
2,e4583023-7353-4a2d-b63f-b8f5772631961759580732708635,e4583023-7353-4a2d-b63f-b8f5772631961759592447843438,20251004,hotel
3,e4583023-7353-4a2d-b63f-b8f5772631961759542335064572,e4583023-7353-4a2d-b63f-b8f5772631961759402462675148,20251004,hotel
4,e4583023-7353-4a2d-b63f-b8f5772631961759592447843438,e4583023-7353-4a2d-b63f-b8f5772631961759572593514300,20251004,hotel
5,BB8F1820-EDE7-4617-A5B5-EEF6D96211531759547394045762,BB8F1820-EDE7-4617-A5B5-EEF6D96211531758126235462625,20251004,hotel
6,cf8bd27c-c47e-4c1e-bab9-443ec25e43d0175956079034534,cf8bd27c-c47e-4c1e-bab9-443ec25e43d01759200230462139,20251004,hotel
7,0F31630D-3C6C-4273-B789-C0E90F3CAB9F1759568839516957,0F31630D-3C6C-4273-B789-C0E90F3CAB9F1759467434164221,20251004,hotel
8,F78CD230-0C2D-4C8A-8DE0-92F7F81351121759581010198480,F78CD230-0C2D-4C8A-8DE0-92F7F81351121759469257143786,20251004,hotel
9,1F1BBB0F-3115-4AC7-B2D4-01D958A9D5C51759584183046697,1F1BBB0F-3115-4AC7-B2D4-01D958A9D5C51759582865801925,20251004,hotel


In [8]:
%%sql test --preview --quiet

select 
    a.*,
    b.survey_intention,
    b.survey_type
from mart_jiulv_flow.dws_stp_session_comp_df a
join mart_hoteldim.stp_survey_result b
on a.session_id = b.session_id

/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column_name] = series
/opt/meituan/spark-3.0/python/pyspark/sql/pandas/conversion.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

,session_id,holiday_weekend_vacation_checkin_datekey_cnt,holiday_checkin_datekey_cnt,weekend_checkin_datekey_cnt,vacation_checkin_datekey_cnt,lead_time,checkin_datekey_cnt,goods_cnt,poi_cnt,aoi_cnt,city_cnt,stay_time,search_page_cnt,search_page_time,poi_list_cnt,poi_list_time,poi_info_cnt,poi_info_time,room_list_cnt,room_list_time,room_info_cnt,room_info_time,create_order_cnt,create_order_time,evaluate_page_cnt,evaluate_page_time,search_scenic_cnt,waitou_cnt,rednote_cnt,order_cnt,star2_cnt,star3_cnt,star4_cnt,star5_cnt,budget_cnt,express_cnt,business_cnt,theme_cnt,couple_cnt,apartment_cnt,inn_cnt,homestay_cnt,hostel_cnt,farmstay_cnt,family_guesthouse_cnt,guesthouse_cnt,resort_hotel_cnt,villa_cnt,family_cnt,esports_cnt,scarce_cnt,unique_cnt,lowprice_cnt,good_cnt,tourism_core_city_cnt,tourism_big_city_cnt,tourism_seasonal_city_cnt,tier1_city_cnt,new_tier1_city_cnt,tier2_city_cnt,tier3_city_cnt,tier4_city_cnt,tier5_city_cnt,university_cnt,transportation_hub_cnt,scenic_area_cnt,hospital_cnt,performance_sports_venue_cnt,convention_center_cnt,industrial_park_cnt,king_room_cnt,single_room_cnt,double_room_cnt,triple_room_cnt,suite_cnt,standalone_cnt,dorm_cnt,is_college_student,is_adult_single,is_adult_married_no_kids,is_adult_married_with_kids,is_over_60,is_senior,is_middle_age,is_young,is_minor,is_high_value,is_mid_value,is_low_value,is_l4,is_l1_3,is_youth_campus,is_business_elite,is_self_social,is_practical_life,is_family_guardian,is_vital_soldier,is_quality_visitor,is_kid_explorer,is_occasional_traveler,holiday_rnt_pct,travel_distance,is_holiday_weekend_vacation,is_holiday,is_weekend,is_vacation,order_lead_time,is_star2,is_star3,is_star4,is_star5,is_budget,is_express,is_business,is_theme,is_couple,is_apartment,is_inn,is_homestay,is_hostel,is_farmstay,is_family_guesthouse,is_guesthouse,is_resort_hotel,is_villa,is_family,is_esports,is_scarce,is_unique,is_lowprice,is_good,order_travel_distance,is_search_scenic,is_waitou,is_rednote,is_tourism_core_city,is_tourism_big_city,is_tourism_seasonal_city,is_tier1_city,is_new_tier1_city,is_tier2_city,is_tier3_city,is_tier4_city,is_tier5_city,is_university,is_transportation_hub,is_scenic_area,is_hospital,is_performance_sports_venue,is_convention_center,is_industrial_park,is_king_room,is_single_room,is_double_room,is_triple_room,is_suite,is_standalone,is_dorm,search_cnt,travel_session_cnt,is_search,is_travel_session,session_cnt,event_cnt,advance_checkin_datekey_cnt,sameday_checkin_datekey_cnt,overnight_checkin_datekey_cnt,dt
0,68c42742-3fd9-4c37-86cc-fcba1c3a87cf1780194675671856,2,0,2,0,0.000000,4,1,1,1,1,38.679,0,0.000,0,0.000,9,20.859,0,0.0,0,0.000,12,15.407,0,0.0,0,0,0,13,22,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22,0,0,22,0,0,0,0,22,0,0,0,22,0,0,0,0,0,0,0,0,13,0,0,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0.074456,0.014047,1,0,1,0,0.000000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,5,22,0,4,0,20260531
1,87765f99-0d3a-4b88-ac43-98c13d9e833a1780193754337666,0,0,0,0,0.000000,1,2,2,1,1,145.249,0,0.000,0,0.000,11,77.323,0,0.0,0,0.000,9,33.267,0,0.0,0,0,0,4,0,0,19,0,0,0,19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,19,0,19,0,19,0,0,0,0,19,0,0,0,0,0,19,0,0,0,8,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0.050000,2.095808,0,0,0,0,0.000000,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,0,1,2,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,24,0,1,0,20260531
2,1a6eac45-99b5-4e06-ac73-9d8fcd9506351777862734723889,1,1,0,0,0.000000,1,8,9,3,1,1499.223,0,0.000,12,291.440,39,542.794,0,0.0,0,0.000,2,15.525,0,0.0,0,0,0,6,62,0,0,0,8,48,6,0,0,0,0,0,0,0,0,0,0,0,9,1,0,8,54,62,62,0,0,0,0,0,0,62,0,0,53,53,1,0,0,0,1,0,18,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0.250000,366.846113,1,1,0,0,0.000000,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,1,1,367,1,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,41,82,1,1,3,89,0,1,0,20260504
3,86e0b3a0-f7cb-411c-aa52-e15f38871a13177737951388742,0,0,0,0,0.000000,1,1,2,1,1,205.26

In [ ]:
test = metrics(test).toPandas()

In [5]:
# test.to_parquet("hdfs:///user/luyiwei03/stp_test.parquet", index=False)
test = pd.read_parquet("hdfs:///user/luyiwei03/stp_test.parquet")

In [6]:
test = test[test['survey_type']=='访谈']

In [10]:
X = test.drop(columns=['intention','session_id','dt','survey_intention','survey_type','level3_biz_code'], errors='ignore')
y_pred_prob_raw = lgbm.predict(X)
y_pred_prob = iso_lgbm.transform(y_pred_prob_raw)
y_pred = (y_pred_prob >= best_threshold_lgbm).astype(int)
y_true = test['survey_intention']

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.7011494252873564
              precision    recall  f1-score   support

           0       0.66      0.72      0.69        40
           1       0.74      0.68      0.71        47

    accuracy                           0.70        87
   macro avg       0.70      0.70      0.70        87
weighted avg       0.71      0.70      0.70        87



In [12]:
X = test.drop(columns=['intention','session_id','dt','survey_intention','survey_type','level3_biz_code'], errors='ignore')
dtest = xgb.DMatrix(X)
y_pred_prob_raw = xgbm.predict(dtest)
y_pred_prob = iso_xgbm.transform(y_pred_prob_raw)
y_pred = (y_pred_prob >= best_threshold_xgbm).astype(int)
y_true = test['survey_intention']

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.6781609195402298
              precision    recall  f1-score   support

           0       0.64      0.68      0.66        40
           1       0.71      0.68      0.70        47

    accuracy                           0.68        87
   macro avg       0.68      0.68      0.68        87
weighted avg       0.68      0.68      0.68        87



In [27]:
X = test.drop(columns=['intention','session_id','dt','survey_intention','survey_type','level3_biz_code'], errors='ignore')
dtest = xgb.DMatrix(X)
lgb_prob_raw = lgbm.predict(X)
lgb_prob_cal = iso_lgbm.transform(lgb_prob_raw)
dtest = xgb.DMatrix(X)
xgb_prob_raw = xgbm.predict(dtest)
xgb_prob_cal = iso_xgbm.transform(xgb_prob_raw)

ensemble_prob = lgb_weight*lgb_prob_cal + xgb_weight*xgb_prob_cal
y_pred = (ensemble_prob >= best_threshold_ensemble).astype(int)
y_true = test['survey_intention']

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.6781609195402298
              precision    recall  f1-score   support

           0       0.64      0.68      0.66        40
           1       0.71      0.68      0.70        47

    accuracy                           0.68        87
   macro avg       0.68      0.68      0.68        87
weighted avg       0.68      0.68      0.68        87

